In [0]:
!pip install emoji

In [0]:
dbutils.library.restartPython()

In [0]:
df=spark.read.option("header", True).csv("/Volumes/amit/bertopic/raw_data/Arabic_offensive_comment_detection_annotation_4000_selected.csv").\
    withColumnRenamed("Id", "id").\
    withColumnRenamed("Comment", "orig_text")

In [0]:
import pyspark.sql.functions as F
df=df.withColumn("incitement_label", F.when(F.col("Majority_Label") == "Non-Offensive","normal").otherwise("abusive")).\
    withColumn("incitement_label", F.when(F.col("Vulgar:V/HateSpeech:HS/None:-") == "HS", "incitement").otherwise(F.col("incitement_label")))




#.write.saveAsTable("amit.bertopic.Arabic_offensive_comment_detection_annotation_input")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
import emoji

# 1. Define the Python function
def replace_emoji_with_text(text):
    if not text:
        return text
    
    # demojize converts 😂 to a string. 
    # language='ar' converts it to Arabic (e.g., :وجه_بدموع_الفرح:)
    # change to language='en' if you prefer English descriptions
    return emoji.demojize(text, language='ar')

# 2. Register it as a PySpark UDF
demojize_udf = F.udf(replace_emoji_with_text, StringType())

In [0]:
clean_df = df.withColumn(
    "text",
    # 1. COLLAPSE REPEATING CHARACTERS (Fixes ههههه, رشااا, etc.)
    F.regexp_replace(F.col("orig_text"), r"(.)\1{2,}", "$1")
).withColumn(
    "text",
    # 2. Translate Emojis (Outputs things like :وجه_مبتسم:)
    demojize_udf(F.col("text"))
).withColumn(
    "text",
    # 3. NEW: COLLAPSE REPEATING EMOJI TAGS
    # Turns :علم_السعودية::علم_السعودية: into just :علم_السعودية:
    F.regexp_replace(F.col("text"), r"(:[^:]+:)\1+", "$1")
).withColumn(
    "text",
    # 4. Remove standard URLs (http, www)
    F.regexp_replace(F.col("text"), r"https?://\S+|www\.\S+", "")
).withColumn(
    "text",
    # 5. Remove the literal <url> placeholders
    F.regexp_replace(F.col("text"), r"(?i)<url>", "")
).withColumn(
    "text",
    # 6. Remove the .IDX and @User.IDX artifacts
    F.regexp_replace(F.col("text"), r"@?User\.IDX|\.IDX", "")
).withColumn(
    "text",
    # 7. Remove English letters (like "XD")
    F.regexp_replace(F.col("text"), r"[a-zA-Z]", "")
).withColumn(
    "text",
    # 8. Clean up emoji underscores/colons
    F.regexp_replace(F.col("text"), r"[:_]", " ")
).withColumn(
    "text",
    # 9. Remove Quotes, ASCII Punctuation, AND Arabic Punctuation (، ؟)
    F.regexp_replace(F.col("text"), r'[!"#\$%&\'\(\)\*\+,\-\./:;<=>\?@\[\\\]\^_`\{\|\}\~،؟]', ' ')
).withColumn(
    "text",
    # 10. Final cleanup of multiple spaces
    F.trim(F.regexp_replace(F.col("text"), r"\s+", " "))
)

In [0]:
clean_df.select("orig_text", "text").limit(30).display()

In [0]:
clean_df.write.saveAsTable("amit.bertopic.Arabic_offensive_comment_detection_annotation_input")